# Comparação pareada e sensibilidade a blocos de calendário
Complemento 03A.05B da Iatan Unit 2. **Sem treinamento e sem consulta à EPA.**
Executar depois de 03A.05A, uma célula por vez na ordem. Usa somente previsões
recebidas e verificadas. Os resultados anteriores não são sobrescritos.

O teste de novembro e dezembro de 2025 já foi consultado. Esta análise é
retrospectiva, condicional aos modelos ajustados e não seleciona um vencedor.
Não estima a variabilidade de inicialização, treino ou escolha do histórico.
Os limites de reamostragem são exploratórios: não são intervalos de previsão,
testes de equivalência ou intervalos com cobertura nominal demonstrada.

In [1]:
import os, sys, json, hashlib, zipfile, io, platform, tempfile
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display

# Local: PROJECT_ROOT; Colab: pasta do projeto no MyDrive.
# Alternativa portátil: enviar o ZIP de insumos ou somente o arquivo que faltar.
IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
default_root = Path('/content/drive/MyDrive/co2-cems-thermal-modeling')
PORTABLE_INPUTS = False
PROJECT = Path(os.environ.get('PROJECT_ROOT', str(default_root)))
if not (PROJECT/'data/interim/03A05_trajectory_history').exists():
    nested = PROJECT/'co2-cems-thermal-modeling'
    if (nested/'data/interim/03A05_trajectory_history').exists():
        PROJECT = nested
if not PROJECT.exists():
    PROJECT = Path(tempfile.mkdtemp(prefix='co2_projeto_'))
    PORTABLE_INPUTS = True
EXPECTED_HASHES = {'data/interim/03A05_trajectory_history/03A05_treino_selecao_A_B.parquet': '7d7aebca5a1d588fe6bff48e25002693724644d0ef5e8836bb408dd1a021080a', 'data/interim/03A05_trajectory_history/03A05_validacao_outubro_2025_A_B.parquet': 'aa54c3b08d43f1151244dbd4d83ebf2673a52a6019c6f8f100190aec6552a54b', 'data/interim/03A05_trajectory_history/03A05_teste_nov_dez_2025_A_B.parquet': '526c80faafd6bd9f22a53197acb90bf06fd8800cbba0443a05c21bf9b9a32c70', 'data/interim/03A05_trajectory_history/03A05_previsoes_pareadas_com_contextos.parquet': 'eb204d81dae4ccc3f073fbe630bd88283bb8a9c3f774dca5bae5b36ac2c699be', 'outputs/03A05A_ablacao_controlada.zip': 'd1f91f7717a7302947fe74e4db839ac4db87219e39281c7ab8e0ce8e7bb2ffa1', 'outputs/03A05A_ablacao_controlada/20261029T235706_270875Z/ablacao_previsoes.parquet': 'f4aeee282d9859d3a9cdc90855f7e0469ad029ee8b227fe9a0ad8af086f021b8'}
ARCHIVE_MEMBERS = {'outputs/03A05A_ablacao_controlada/20261029T235706_270875Z/ablacao_previsoes.parquet': {'archive': 'outputs/03A05A_ablacao_controlada.zip', 'member': '03A05A_ablacao_controlada/20261029T235706_270875Z/ablacao_previsoes.parquet'}}
INPUT_FILES = {}
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def input_path(relative):
    return INPUT_FILES.get(relative, PROJECT/relative)

def archive_payload(data, relative):
    if not zipfile.is_zipfile(io.BytesIO(data)):
        return None
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        # Pacote portátil com arquivos em seus caminhos relativos.
        names = [n for n in z.namelist() if n == relative or n.endswith('/'+relative)]
        if len(names) == 1:
            return z.read(names[0])
        mapping = ARCHIVE_MEMBERS.get(relative)
        if mapping:
            # ZIP original da ablação, sem prefixo outputs/.
            if mapping['member'] in z.namelist():
                return z.read(mapping['member'])
            # Compatibilidade com a primeira entrega do pacote de insumos.
            if mapping['archive'] in z.namelist():
                return archive_payload(z.read(mapping['archive']), relative)
    return None

def cache_input(relative, payload):
    assert hashlib.sha256(payload).hexdigest() == EXPECTED_HASHES[relative], 'Arquivo diferente do insumo conferido: '+relative
    destination = Path(tempfile.mkdtemp(prefix='co2_insumo_'))/relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(payload)
    INPUT_FILES[relative] = destination

def validate_inputs(paths):
    global PORTABLE_INPUTS
    # Conferir TODOS os arquivos necessários, mesmo se a pasta data já existir.
    # Um ZIP ausente não implica refazer o experimento que o produziu.
    roots = [PROJECT, PROJECT/'co2-cems-thermal-modeling', default_root,
             default_root/'co2-cems-thermal-modeling']
    missing = []
    for relative in paths:
        candidates = [input_path(relative)] + [root/relative for root in roots]
        if IN_COLAB:
            candidates.append(Path('/content')/Path(relative).name)
        found = next((p for p in candidates if p.is_file()), None)
        # Prioridade para os arquivos que o 5A salva diretamente no Drive.
        # O ZIP é só uma alternativa de transporte, não uma saída obrigatória.
        if found is None and relative in ARCHIVE_MEMBERS:
            mapping = ARCHIVE_MEMBERS[relative]
            archives = [root/mapping['archive'] for root in roots]
            if IN_COLAB:
                archives.append(Path('/content')/Path(mapping['archive']).name)
            for archive in archives:
                if archive.is_file():
                    payload = archive_payload(archive.read_bytes(),relative)
                    if payload is not None:
                        cache_input(relative,payload)
                        found = INPUT_FILES[relative]
                        break
        if found is None:
            missing.append(relative)
        else:
            INPUT_FILES[relative] = found
    if missing:
        print('Faltam estes arquivos:', *missing, sep='\n  ')
        if not IN_COLAB:
            raise FileNotFoundError('Insumos ausentes: '+', '.join(missing))
        from google.colab import files
        print('Envie 03A05BC_insumos.zip OU os arquivos ausentes já gerados. Não reexecute o 5A.')
        uploaded = files.upload()
        for relative in missing:
            payload = None
            for filename, data in uploaded.items():
                # Também aceita diretamente o ZIP da ablação, mesmo se o navegador
                # tiver acrescentado (1) ao nome. A identidade vem do SHA-256.
                if hashlib.sha256(data).hexdigest() == EXPECTED_HASHES[relative]:
                    payload = data
                    break
                payload = archive_payload(data,relative)
                if payload is not None:
                    break
            if payload is None:
                raise FileNotFoundError('O envio não contém '+relative+'. Envie 03A05BC_insumos.zip.')
            cache_input(relative,payload)
        PORTABLE_INPUTS = True  # Oferecer download final também quando houver upload.
    actual = {p: sha256(input_path(p)) for p in paths}
    for p,h in actual.items():
        assert h == EXPECTED_HASHES[p], f'Insumo diferente da execução recebida: {p}'
    return actual
BASE = PROJECT/'data/interim/03A05_trajectory_history'
TARGET = 'CO2 Mass (short tons)'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')

Mounted at /content/drive


## Protocolo fixado antes desta execução complementar
Perda absoluta em short tons por registro horário. Para cada par, d(t) =
|y(t) − previsão da referência| − |y(t) − previsão da candidata|. Valor positivo
favorece a candidata. Comparar B completo com A; três reduções de B com B
completo; e cada redução com A. Todos os sete pares e todos os tamanhos de
bloco são publicados, sem escolha pelo sinal dos limites.

Blocos não sobrepostos de **1, 3 e 7 dias corridos**, ancorados em 01/11/2025,
com último bloco parcial. Calendário completo até 31/12/2025, inclusive dias
sem observações elegíveis. Reamostrar blocos inteiros com reposição, mantendo
pares e contextos juntos; calcular a razão entre somas de diferenças e contagens
ON em cada réplica. **10.000 réplicas, semente 42**, mesmos sorteios para todos
os pares e contextos em cada tamanho. Não confundir dias com 24 linhas ON.

Os contextos são os já salvos; não redefinir o limiar de rampa de 56 MW.
Sobreposição partida/rampa (uma hora) é apenas descritiva. Para partidas,
reportar a concentração em poucos blocos e réplicas sem observações; tais
limites são especialmente frágeis. A escolha de comprimentos é sensibilidade
própria do estudo, não uma aplicação de garantia assintótica de Künsch.

In [2]:
INPUT_PATHS = [
    'data/interim/03A05_trajectory_history/03A05_previsoes_pareadas_com_contextos.parquet',
    'outputs/03A05A_ablacao_controlada/20261029T235706_270875Z/ablacao_previsoes.parquet']
INPUT_HASHES = validate_inputs(INPUT_PATHS)
original = pd.read_parquet(input_path(INPUT_PATHS[0])).sort_values('timestamp').reset_index(drop=True)
ab = pd.read_parquet(input_path(INPUT_PATHS[1])).sort_values('timestamp').reset_index(drop=True)
assert len(original) == len(ab) == 1144
assert original.timestamp.is_unique and ab.timestamp.is_unique
assert np.array_equal(original.timestamp.to_numpy(), ab.timestamp.to_numpy())
assert np.array_equal(original[TARGET].to_numpy(), ab[TARGET].to_numpy())
assert np.array_equal(original.pred_B.to_numpy(), ab.pred_B_completo.to_numpy())
df = original.copy()
for c in ['pred_B_completo','pred_B_sem_media3h','pred_B_sem_delta','pred_B_sem_delta_e_absdelta']:
    df[c] = ab[c].to_numpy()
assert df.timestamp.min() == pd.Timestamp('2025-11-01 00:00:00')
assert df.timestamp.max() == pd.Timestamp('2025-12-31 23:00:00')
assert np.isfinite(df[[TARGET,'pred_A'] + [c for c in ab if c.startswith('pred_B_')]].to_numpy()).all()
for c in ['startup_first_6h','intense_ramp','regular_other']:
    assert np.array_equal(df[c].to_numpy().astype(bool), ab[c].to_numpy().astype(bool))
CONTEXTS = {'global':np.ones(len(df),bool), 'partidas':df.startup_first_6h.to_numpy(bool),
            'rampas':df.intense_ramp.to_numpy(bool), 'demais':df.regular_other.to_numpy(bool),
            'sobreposicao':df.startup_and_ramp.to_numpy(bool)}
assert [int(v.sum()) for v in CONTEXTS.values()] == [1144,30,258,857,1]
PAIRS = [('A','B_completo')] + [('B_completo',v) for v in
        ['B_sem_media3h','B_sem_delta','B_sem_delta_e_absdelta']] + [('A',v) for v in
        ['B_sem_media3h','B_sem_delta','B_sem_delta_e_absdelta']]
PREDS = {'A':'pred_A', **{v:'pred_'+v for v in ['B_completo','B_sem_media3h','B_sem_delta','B_sem_delta_e_absdelta']}}
y = df[TARGET].to_numpy(dtype='float64')
errors = {k: np.abs(y-df[v].to_numpy(dtype='float64')) for k,v in PREDS.items()}
metrics = []
for name,c in PREDS.items():
    for ctx,mask in CONTEXTS.items():
        residual = df[c].to_numpy(dtype='float64')[mask]-y[mask]
        metrics.append(dict(modelo=name,contexto=ctx,n=int(mask.sum()),
            MAE=float(np.abs(residual).mean()),RMSE=float(np.sqrt(np.mean(residual**2))),
            vies_observado_menos_pred=float(-residual.mean())))
metrics = pd.DataFrame(metrics)
display(metrics[metrics.contexto.isin(['global','partidas','rampas','demais'])].round(6))

,modelo,contexto,n,MAE,RMSE,vies_observado_menos_pred
0,A,global,1144,20.151419,25.065266,-10.346744
1,A,partidas,30,11.728783,16.353259,-10.127421
2,A,rampas,258,24.057700,29.360946,-17.902829
3,A,demais,857,19.254376,23.871129,-8.075204
5,B_completo,global,1144,21.171987,26.577858,-2.550333
6,B_completo,partidas,30,10.451707,13.165937,-8.533026
7,B_completo,rampas,258,18.549117,22.480886,-3.588153
8,B_completo,demais,857,22.341564,28.026185,-2.054886
10,B_sem_media3h,global,1144,20.689602,25.858607,-3.987032
11,B_sem_media3h,partidas,30,10.239119,13.168128,-8.244002


In [3]:
START, END = pd.Timestamp('2025-11-01'), pd.Timestamp('2026-01-01')
REPLICATES, SEED, BLOCK_DAYS = 10000, 42, [1,3,7]
calendar = pd.date_range(START, END, freq='h', inclusive='left')
assert len(calendar) == 1464
assert df.timestamp.isin(calendar).all()
bootstrap_rows, block_rows, draws = [], [], {}
for days in BLOCK_DAYS:
    nblocks = int(np.ceil((END-START)/pd.Timedelta(days=days)))
    block = ((df.timestamp-START)/pd.Timedelta(days=days)).astype(int).to_numpy()
    # Sorteios compartilhados; a sequência de cada tamanho independe dos demais.
    rng = np.random.default_rng(np.random.SeedSequence([SEED,days]))
    sampled = rng.integers(0,nblocks,size=(REPLICATES,nblocks))
    draws[f'indices_blocos_{days}d'] = sampled
    for ctx,mask in CONTEXTS.items():
        count = np.bincount(block, weights=mask.astype(int), minlength=nblocks)
        for k,n in enumerate(count):
            lo=START+pd.Timedelta(days=k*days); hi=min(lo+pd.Timedelta(days=days),END)
            block_rows.append(dict(bloco_dias=days,contexto=ctx,bloco=k,inicio=str(lo),
                fim_exclusivo=str(hi),horas_calendario=int((hi-lo)/pd.Timedelta(hours=1)),n=int(n)))
        denom = count[sampled].sum(axis=1)
        for reference,candidate in PAIRS:
            d = errors[reference]-errors[candidate]
            sums = np.bincount(block, weights=np.where(mask,d,0),minlength=nblocks)
            assert np.isclose(sums.sum()/count.sum(),d[mask].mean(),rtol=0,atol=1e-12)
            if ctx == 'sobreposicao':
                low=high=float('nan'); valid=0
            else:
                valid_mask = denom>0
                values = sums[sampled].sum(axis=1)[valid_mask]/denom[valid_mask]
                assert np.isfinite(values).all()
                low,high = np.quantile(values,[.025,.975]); valid=int(valid_mask.sum())
            bootstrap_rows.append(dict(referencia=reference,candidata=candidate,contexto=ctx,
                bloco_dias=days,n=int(mask.sum()),n_blocos=nblocks,
                blocos_com_observacoes=int((count>0).sum()),d_medio=float(d[mask].mean()),
                ganho_mae_pct=float(100*d[mask].mean()/errors[reference][mask].mean()),
                percentil_2_5=float(low),percentil_97_5=float(high),replicas_validas=valid,
                replicas_sem_contexto=(int((denom==0).sum()) if ctx!='sobreposicao' else None),
                inferencia='apenas descritivo n=1' if ctx=='sobreposicao' else 'sensibilidade exploratoria; cobertura nao demonstrada'))
intervals = pd.DataFrame(bootstrap_rows)
blocks = pd.DataFrame(block_rows)
display(intervals[(intervals.referencia=='A') & (intervals.candidata=='B_completo')].round(6))

,referencia,candidata,contexto,bloco_dias,n,n_blocos,blocos_com_observacoes,d_medio,ganho_mae_pct,percentil_2_5,percentil_97_5,replicas_validas,replicas_sem_contexto,inferencia
0,A,B_completo,global,1,1144,61,53,-1.020568,-5.064495,-3.048488,0.849060,10000,0.0,sensibilidade exploratoria; cobertura nao demo...
7,A,B_completo,partidas,1,30,61,7,1.277076,10.888396,-1.584903,4.221179,9995,5.0,sensibilidade exploratoria; cobertura nao demo...
14,A,B_completo,rampas,1,258,61,47,5.508583,22.897380,3.109778,8.079483,10000,0.0,sensibilidade exploratoria; cobertura nao demo...
21,A,B_completo,demais,1,857,61,50,-3.087188,-16.033693,-5.604393,-0.673849,10000,0.0,sensibilidade exploratoria; cobertura nao demo...
28,A,B_completo,sobreposicao,1,1,61,1,-18.663727,-285.943187,NaN,NaN,0,NaN,apenas descritivo n=1
35,A,B_completo,global,3,1144,21,19,-1.020568,-5.064495,-3.858398,1.411710,10000,0.0,sensibilidade exploratoria; cobertura nao demo...
42,A,B_completo,partidas,3,30,21,5,1.277076,10.888396,-0.036861,2.825853,9961,39.0,sensibilidade exploratoria; cobertura nao demo...
49,A,B_completo,rampas,3,258,21,19,5.508583,22.897380,2.901103,8.313191,10000,0.0,sensibilidade exploratoria; cobertura nao demo...
56,A,B_completo,demais,3,857,21,19,-3.087188,-16.033693,-6.547404,0.100661,10000,0.0,sensibilidade exploratoria; cobertura nao demo...
63,A,B_completo,sobreposicao,3,1,21,1,-18.663727,-285.943187,NaN,NaN,0,NaN,apenas descritivo n=1


In [4]:
# Correlação em horas de calendário: OFF/inelegíveis ficam NaN, não viram zero.
# Correlacionar os pares válidos; n_pares varia com a defasagem.
acf_rows=[]
for reference,candidate in PAIRS:
    s = pd.Series(errors[reference]-errors[candidate], index=pd.DatetimeIndex(df.timestamp)).reindex(calendar)
    for lag in [1,3,6,12,24,48,168]:
        earlier=s.shift(lag); valid=s.notna() & earlier.notna()
        acf_rows.append(dict(referencia=reference,candidata=candidate,lag_horas=lag,
            n_pares=int(valid.sum()),correlacao_pares_validos=float(s[valid].corr(earlier[valid]))))
acf = pd.DataFrame(acf_rows)
display(acf[(acf.referencia=='A') & (acf.candidata=='B_completo')])

,referencia,candidata,lag_horas,n_pares,correlacao_pares_validos
0,A,B_completo,1,1138,0.441207
1,A,B_completo,3,1126,0.219642
2,A,B_completo,6,1109,0.084059
3,A,B_completo,12,1079,0.108571
4,A,B_completo,24,1019,0.124305
5,A,B_completo,48,908,0.142751
6,A,B_completo,168,837,0.082543


In [5]:
# Publicar só no diretório exclusivo desta execução. Os insumos não são alterados.
OUTPUT = PROJECT/'outputs'/'03A05B_incerteza_pareada'/RUN_ID
OUTPUT.mkdir(parents=True,exist_ok=False)
metrics.to_csv(OUTPUT/'metricas_contextos.csv',index=False)
intervals.to_csv(OUTPUT/'sensibilidade_blocos_calendario.csv',index=False)
blocks.to_csv(OUTPUT/'contagens_blocos.csv',index=False)
acf.to_csv(OUTPUT/'autocorrelacao_calendario.csv',index=False)
np.savez_compressed(OUTPUT/'sorteios_blocos.npz',**draws)
assert validate_inputs(INPUT_PATHS) == INPUT_HASHES
protocol = dict(natureza='retrospectiva condicional aos modelos salvos; sem treinamento',
    perda='absoluta; d=erro absoluto referencia - candidata',semente=SEED,replicas=REPLICATES,
    blocos_dias=BLOCK_DAYS,ancora=str(START),fim_exclusivo=str(END),horas_calendario=len(calendar),
    dias=61,n_ON=len(df),pares=PAIRS,contextos={k:int(v.sum()) for k,v in CONTEXTS.items()},
    intervalos='percentis exploratorios 2.5 e 97.5; sem cobertura nominal demonstrada',
    exclusoes_inferencia=['sobreposicao n=1'],teste_reutilizado=True,novos_ajustes=0,
    hashes_insumos=INPUT_HASHES,python=sys.version,platform=platform.platform(),numpy=np.__version__,pandas=pd.__version__)
(OUTPUT/'protocolo.json').write_text(json.dumps(protocol,ensure_ascii=False,indent=2),encoding='utf-8')
import shutil
archive=shutil.make_archive(str(OUTPUT.parent/('03A05B_'+RUN_ID)),'zip',root_dir=OUTPUT)
print('Concluído. Resultados:',OUTPUT)
print('Arquivo para compartilhar:',archive)
if PORTABLE_INPUTS:
    from google.colab import files
    files.download(archive)
# No Colab, o ZIP fica na pasta outputs/03A05B_incerteza_pareada do projeto.

Concluído. Resultados: /content/drive/MyDrive/co2-cems-thermal-modeling/outputs/03A05B_incerteza_pareada/20261030T015541_758746Z
Arquivo para compartilhar: /content/drive/MyDrive/co2-cems-thermal-modeling/outputs/03A05B_incerteza_pareada/03A05B_20261030T015541_758746Z.zip


## Referências e alcance
DIEBOLD, F. X.; MARIANO, R. S. Comparing Predictive Accuracy. NBER Technical
Working Paper 169, 1994. https://doi.org/10.3386/t0169.
Fundamenta a comparação pareada de perdas; **não executamos o teste DM/HAC**.

KÜNSCH, H. R. The jackknife and the bootstrap for general stationary observations.
The Annals of Statistics, 17(3), 1217–1241, 1989. https://doi.org/10.1214/aos/1176347265.
Referência geral para blocos sob dependência; o calendário irregular, o curto
período e a seleção retrospectiva aqui não satisfazem automaticamente suas
garantias. O desenho de blocos é adaptação própria, apresentada como sensibilidade.

A interpretação deve considerar os três comprimentos e a quantidade de blocos
com observações. Nem um limite que exclua zero resolve o teste reutilizado,
mudanças de regime, seleção de modelos ou multiplicidade das comparações.